In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import json

###Fase 1: Data Engineering

Abran el archivo .json usando la librería json de Python.
Extraigan los datos para construir una única tabla maestra (DataFrame) con 3 columnas: Origen, Destino y Relacion.
1. Reto A (La Capa Social): Extraigan los "retweets". Si Usuario_A retuitea a Usuario_B, eso es una relación tipo "Retweet". Ignoren auto-retweets. (Ojo con las minúsculas/mayúsculas).
2. Reto B (La Capa de Infraestructura): En el JSON, la IP y la Tarjeta son metadatos individuales. Deberán ingeniárselas (recomendación: extraigan los metadatos a un DataFrame y hagan un merge del dataframe consigo mismo) para conectar a los usuarios. Si Usuario_A y Usuario_B tienen la misma Tarjeta, agreguen una fila a su tabla con relación tipo "Misma_Tarjeta".


In [ ]:
with open("api_troll_farm.json", "r", encoding="utf-8") as archivo:
    data_json = json.load(archivo)

#  RETO A: Lista para guardar los registros de retweets procesados
filas_retweet = []

for registro in data_json:
    usuario_origen = str(registro.get('account_id', '')).strip().lower()
    activity = registro.get('activity', {})

    if isinstance(activity, dict) and 'retweets' in activity:
        retweets = activity['retweets']
        if isinstance(retweets, list):
            for usuario_destino in retweets:
                usuario_destino_clean = str(usuario_destino).strip().lower()

                # Evitar auto-retweets y asegurar nombres de usuario válidos
                if usuario_destino_clean and usuario_origen != usuario_destino_clean:
                    filas_retweet.append({
                        'Origen': usuario_origen,
                        'Destino': usuario_destino_clean,
                        'Relacion': 'Retweet'
                    })

df_retweets = pd.DataFrame(filas_retweet, columns=['Origen', 'Destino', 'Relacion'])

df = df_retweets

display(df.head())
print(f'Relaciones de Retweet procesadas: {len(df)}')

,Origen,Destino,Relacion
0,troll_bot_1,real_user_11,Retweet
1,real_user_14,real_user_15,Retweet
2,real_user_14,real_user_11,Retweet
3,real_user_13,real_user_3,Retweet
4,real_user_13,real_user_9,Retweet


Relaciones de Retweet procesadas: 37


In [ ]:
### Reto B
# extraer metadatos de los usuarios
filas_metadata = []

for registro in data_json:
    usuario = str(registro.get('account_id', '')).strip().lower()
    metadata = registro.get('metadata') or {}

    if isinstance(metadata, dict):
        filas_metadata.append({
            'Usuario': usuario,
            'IP': metadata.get('last_ip'),
            'Tarjeta': metadata.get('payment_hash')
        })

df_metadata = pd.DataFrame(filas_metadata)

# evitar usuarios duplicados
df_metadata = df_metadata.drop_duplicates(subset=['Usuario'])

display(df_metadata.head())



,Usuario,IP,Tarjeta
0,troll_bot_1,104.28.19.112,VISA-4455-XXXX-9090
1,real_user_14,189.168.1.196,MC-8573-XXXX-1111
2,real_user_13,189.168.1.184,None
3,troll_bot_5,104.28.19.112,VISA-4455-XXXX-9090
4,real_user_12,189.168.1.51,MC-4432-XXXX-1111


In [ ]:
# funcion para crear relaciones entre usuarios
def crear_relaciones(df, columna, tipo_relacion):

    datos = df[['Usuario', columna]].dropna()

    # unir el DataFrame consigo mismo
    combinaciones = datos.merge(datos, on=columna)

    # evitar conexiones del usuario consigo mismo
    # y relaciones duplicadas
    combinaciones = combinaciones[
        combinaciones['Usuario_x'] < combinaciones['Usuario_y']
    ]

    relaciones = pd.DataFrame({
        'Origen': combinaciones['Usuario_x'],
        'Destino': combinaciones['Usuario_y'],
        'Relacion': tipo_relacion
    })

    return relaciones


# relaciones por IP
df_ip = crear_relaciones(df_metadata, 'IP', 'Misma_IP')

# relaciones por tarjeta
df_tarjetas = crear_relaciones(df_metadata, 'Tarjeta', 'Misma_Tarjeta')

display(df_ip.head())
display(df_tarjetas.head())

,Origen,Destino,Relacion
1,troll_bot_1,troll_bot_5,Misma_IP
2,troll_bot_1,troll_bot_4,Misma_IP
3,troll_bot_1,troll_bot_6,Misma_IP
4,troll_bot_1,troll_bot_3,Misma_IP
5,troll_bot_1,troll_bot_2,Misma_IP


,Origen,Destino,Relacion
1,troll_bot_1,troll_bot_5,Misma_Tarjeta
2,troll_bot_1,troll_bot_4,Misma_Tarjeta
3,troll_bot_1,troll_bot_6,Misma_Tarjeta
4,troll_bot_1,troll_bot_3,Misma_Tarjeta
5,troll_bot_1,troll_bot_2,Misma_Tarjeta


In [ ]:
# uniendo todas las relaciones
df_maestro = pd.concat(
    [df_retweets, df_ip, df_tarjetas],
    ignore_index=True
)

display(df_maestro.head(10))

print("Total de relaciones:", len(df_maestro))

print("\nRelaciones por tipo:")
print(df_maestro['Relacion'].value_counts())

,Origen,Destino,Relacion
0,troll_bot_1,real_user_11,Retweet
1,real_user_14,real_user_15,Retweet
2,real_user_14,real_user_11,Retweet
3,real_user_13,real_user_3,Retweet
4,real_user_13,real_user_9,Retweet
5,troll_bot_5,real_user_12,Retweet
6,troll_bot_5,troll_bot_1,Retweet
7,real_user_12,real_user_15,Retweet
8,real_user_11,real_user_14,Retweet
9,real_user_11,real_user_6,Retweet


Total de relaciones: 67

Relaciones por tipo:
Relacion
Retweet          37
Misma_IP         15
Misma_Tarjeta    15
Name: count, dtype: int64


###Fase 2: El Grafo Multicapa

Pasen su tabla maestra purificada a NetworkX utilizando nx.MultiDiGraph().
Comprueben que su grafo cargó correctamente imprimiendo el número total de Nodos y Aristas.

In [ ]:
G = nx.MultiDiGraph()

# agregando los nodos
for registro in data_json:
    usuario = str(registro.get('account_id', '')).strip().lower()
    G.add_node(usuario)

# agregando las aristas con la sintaxis correcta
for _, fila in df_maestro.iterrows():
    G.add_edge(
        fila['Origen'],
        fila['Destino'],
        Relacion=fila['Relacion']
    )

print("Total de nodos:", G.number_of_nodes())
print("Total de aristas:", G.number_of_edges())

Total de nodos: 22
Total de aristas: 67


###Fase 3: El Slicing Los trolls son lo suficientemente listos para esconderse en el ruido de los retweets, pero cometieron un error de novatos: todos pagaron su cuenta bot con la misma tarjeta de crédito robada.

Filtren su MultiGrafo para extraer exclusivamente las aristas cuya Relación sea "Misma_Tarjeta".
Metan esas aristas en un grafo no dirigido simple (nx.Graph()).
Usen la función nx.connected_components() sobre este nuevo sub-grafo para encontrar a la tribu coludida financieramente.

In [ ]:
G_tarjetas = nx.Graph()

for origen, destino, datos in G.edges(data=True):

    if datos['Relacion'] == 'Misma_Tarjeta':
        G_tarjetas.add_edge(origen, destino)

print("Nodos conectados por tarjeta:", G_tarjetas.number_of_nodes())
print("Aristas por tarjeta:", G_tarjetas.number_of_edges())

Nodos conectados por tarjeta: 6
Aristas por tarjeta: 15


In [ ]:
# encontrando componentes conectados
componentes = list(nx.connected_components(G_tarjetas))

print("Componentes conectados encontrados:")

for i, componente in enumerate(componentes, 1):
    print(f"\nGrupo {i}:")
    print(componente)
    print("Cantidad de usuarios:", len(componente))

Componentes conectados encontrados:

Grupo 1:
{'troll_bot_5', 'troll_bot_6', 'troll_bot_2', 'troll_bot_4', 'troll_bot_3', 'troll_bot_1'}
Cantidad de usuarios: 6


In [ ]:
grupo_bots = max(componentes, key=len)

tarjeta_bloqueada = df_metadata[
    df_metadata['Usuario'].isin(grupo_bots)
]['Tarjeta'].mode()[0]

print("BOTS IDENTIFICADOS:")

for bot in sorted(grupo_bots):
    print(bot)

print("\nTarjeta comprometida:", tarjeta_bloqueada)
print("Total de bots:", len(grupo_bots))

BOTS IDENTIFICADOS:
troll_bot_1
troll_bot_2
troll_bot_3
troll_bot_4
troll_bot_5
troll_bot_6

Tarjeta comprometida: VISA-4455-XXXX-9090
Total de bots: 6


# **Veredicto**
### 1 ¿Cuáles son los nombres exactos (account_id) de los bots que conforman el anillo de fraude?
Los nombres exactos son troll_bot_1, troll_bot_2, troll_bot_3, troll_bot_4, troll_bot_5, troll_bot_6 son los que conforman el anillo de fraude

### 2 ¿Cuál es el Hash de la tarjeta de crédito comprometida que el equipo de soporte técnico debe bloquear ahora mismo?

La tarjeta de credito comprometida es la sigueinte: VISA-4455-XXXX-9090, se deeb cancelar a la brevedad

### 3 Si tuvieran que entrenar un modelo de predicción, ¿cómo le pasarían este descubrimiento al modelo tabular (La matriz X de entrenamiento)?

Se pasaria como un patron en donde pueda ser capaz de identificar el patron de fraude y poder predecir si es un posible fraude, esto primero analizando si hay mucihas cuentas usando una misma tarjeta, etc. Esto se pdoria lograr agregando ua columa nueva de cuentas_por_tarjeta, otra columna para usar metricas de centralidad o pagerank, y una variable binaria que indique si pertence a un grupo de fraude o no.
